# Problems 7 - Principal Component Analysis (SOLUTIONS)

## Classifying Flowers

The Iris Dataset is a well known database in the pattern recognition literature. The data set contains 3 classes of 50 instances each, where each class refers to a type of iris plant.

Four features are given:
1. sepal length in cm 
2. sepal width in cm 
3. petal length in cm 
4. petal width in cm 

We will explore this dataset next.

In [ ]:
# Download the helper modules from the Session 7 lab (run once)
import urllib.request

BASE = "https://raw.githubusercontent.com/cienciadadesdsuab/DS26-27/main/Laboratoris/Session%207%20-%20PCA/"
for f in ["linear_algebra.py", "gradient_descent.py"]:
    urllib.request.urlretrieve(BASE + f, f)

import numpy as np
import tqdm


In [ ]:
from sklearn import datasets

iris = datasets.load_iris()
X = iris.data
y = iris.target

print(X[:10])
print(y[:10])

list(iris.target_names)

In [ ]:
print("X.shape: ", X.shape)
m, n = X.shape
print("# Samples: ", m)
print("# Features: ", n)



### PCA Calculation

Let's apply PCA manually to reduce the dataset to only 2 components, using the functions from the lab (`de_mean`, `direction`, `directional_variance`, `directional_variance_gradient`, ... etc). Copy them from the Session 7 lab notebook into this one. Remember the steps:  

**1)** Find the direction with maximum variance → first principal component 

**2)** Project the data onto that direction 

**3)** Remove that direction from the data (the explained part) 

**4)** Repeat to find the next directions  

**Please don't use sklearn to solve this task** (you can use it in the last section, to assess the accuracy) 

**Hints** = You will need to convert the array X to a list of vectors, you can achieve that using **X.tolist()**

**Solution.** The building blocks are the functions from the lab.

In [ ]:
from typing import List
from linear_algebra import Vector, subtract, vector_mean, dot, magnitude, scalar_multiply
from gradient_descent import gradient_step

def de_mean(data: List[Vector]) -> List[Vector]:
    """Recenters the data to have mean 0 in every dimension"""
    mean = vector_mean(data)
    return [subtract(vector, mean) for vector in data]

def direction(w: Vector) -> Vector:
    mag = magnitude(w)
    return [w_i / mag for w_i in w]

def directional_variance(data: List[Vector], w: Vector) -> float:
    w_dir = direction(w)
    return sum(dot(v, w_dir) ** 2 for v in data)

def directional_variance_gradient(data: List[Vector], w: Vector) -> Vector:
    w_dir = direction(w)
    return [sum(2 * dot(v, w_dir) * v[i] for v in data) for i in range(len(w))]

def first_principal_component(data: List[Vector], n: int = 100, step_size: float = 0.1) -> Vector:
    guess = [1.0 for _ in data[0]]
    with tqdm.trange(n) as t:
        for _ in t:
            dv = directional_variance(data, guess)
            gradient = directional_variance_gradient(data, guess)
            guess = gradient_step(guess, gradient, step_size)
            t.set_description(f"dv: {dv:.3f}")
    return direction(guess)

def project(v: Vector, w: Vector) -> Vector:
    return scalar_multiply(dot(v, w), w)

def remove_projection_from_vector(v: Vector, w: Vector) -> Vector:
    return subtract(v, project(v, w))

def remove_projection(data: List[Vector], w: Vector) -> List[Vector]:
    return [remove_projection_from_vector(v, w) for v in data]

def pca(data: List[Vector], num_components: int) -> List[Vector]:
    components: List[Vector] = []
    for _ in range(num_components):
        component = first_principal_component(data)
        components.append(component)
        data = remove_projection(data, component)
    return components

def transform_vector(v: Vector, components: List[Vector]) -> Vector:
    return [dot(v, w) for w in components]

def transform(data: List[Vector], components: List[Vector]) -> List[Vector]:
    return [transform_vector(v, components) for v in data]

In [ ]:
data = X.tolist()                       # array -> list of vectors
centred = de_mean(data)                 # 1) centre every column
components = pca(centred, 2)            # 2-4) first two principal components
reduced_x = transform(centred, components)   # new coordinates: m rows x 2 columns

print("Components:")
for w in components:
    print([round(c, 3) for c in w])
print("reduced_x has", len(reduced_x), "rows and", len(reduced_x[0]), "columns")

Check: how much of the total variance do the 2 components capture?

In [ ]:
total = sum(dot(v, v) for v in centred)
for k, w in enumerate(components, start=1):
    print(f"PC{k} explains {directional_variance(centred, w) / total:.1%} of the variance")

## Assess the accuracy of the model using reduced components:

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
X_train, X_test, y_train, y_test = train_test_split(reduced_x, y, test_size=0.2, random_state=30)

model = LogisticRegression(max_iter=2000)
model.fit(X_train, y_train)
model.score(X_test, y_test)

**Result.** With only 2 components (about 97.8% of the variance) the accuracy on the test set is 0.967, the same as with the 4 original features: almost all the information in the flowers is in two directions.

**Discussion.** Below we compare the 2-component model with one trained on all 4 original features, and our components with scikit-learn's PCA (very close; the manual one runs only 100 gradient steps and may differ in the third decimal, and signs can be flipped).

In [ ]:
from sklearn.decomposition import PCA

# All 4 original features
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=30)
full = LogisticRegression(max_iter=2000).fit(Xtr, ytr)
print("Accuracy with the 4 original features:", round(full.score(Xte, yte), 3))

# scikit-learn PCA
pca_sk = PCA(n_components=2).fit(X)
print("sklearn components:\n", pca_sk.components_.round(3))
print("sklearn explained variance ratio:", pca_sk.explained_variance_ratio_.round(3))